# SQL Practice — Executed For Real, Against Your Loans Dataset

This sets up an in-memory SQL database from `loans_dataset.csv` so every query below actually runs against the same messy real data you've been exploring in pandas — not a toy `employees` table. This is the direct rehearsal for what I think is the most likely real format: you explore in Python, then translate to SQL on demand for the same case.

**How to use this:** for each section, read the "how to think" paragraph, try writing the query yourself in a scratch cell first, *then* run the solution cell and compare. Don't just read top to bottom — that defeats the point.

## Setup — load the CSV into an in-memory SQL database

**Why `sqlite3`:** it's in the Python standard library — nothing to install, which matters if you're on an unfamiliar machine/environment live. `pandas.DataFrame.to_sql()` pushes your dataframe straight into a real SQL table you can query with real SQL. (If `duckdb` happens to be available in the real session, it has friendlier date-handling syntax — same idea, swap the connection.)

A tiny `q()` helper wrapping `pd.read_sql_query` turns every query into a one-liner — small thing, but setting yourself up for speed is exactly the kind of habit that reads as experienced.

In [8]:
import pandas as pd
import sqlite3

df = pd.read_csv('loans_dataset.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('loans', conn, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql_query(sql, conn)

q("SELECT COUNT(*) AS total_rows FROM loans")

,total_rows
0,1454


## 1. First look in SQL

**How to think about it:** the SQL equivalent of `.shape` / `.head()` / `.nunique()` is `COUNT(*)`, `LIMIT`, and `COUNT(DISTINCT ...)`. Comparing `COUNT(*)` against `COUNT(DISTINCT loan_id)` in the *same query* immediately flags duplicates — you don't need a separate step to notice something's off.

In [21]:
q('''
SELECT *
FROM loans;
''')

,loan_id,customer_id,loan_amount,tenor_months,risk_tier,acquisition_channel,city,is_repeat_borrower,repayment_status,disbursed_at
0,500695,C-00052,296000.0,3,C,REFERRAL,Makassar,NaN,ON_TIME,2026-06-05T18:10:00
1,500372,C00015,517000.0,3,B,ADS,makassar,TRUE,ON_TIME,2026-05-26T15:38:00
2,500221,C-00302,330000.0,6,B,PARTNER_MERCHANT,medan,NaN,ON_TIME,2026-05-21T15:43:00
3,501086,C-00327,701000.0,3,A,APP,Bandung,TRUE,ON_TIME,2026-06-17T16:24:00
4,501290,C-00273,743000.0,6,D,REFERRAL,Bandung,TRUE,ON_TIME,2026-06-23T05:00:00
...,...,...,...,...,...,...,...,...,...,...
1449,500333,C00279,409000.0,6,D,APP,medan,1,ON_TIME,2026-05-25T22:56:00
1450,501294,C00201,311000.0,3,B,ADS,Bandung,FALSE,ON_TIME,2026-06-23T06:01:00
1451,501362,C00155,614000.0,12,B,REFERRAL,makassar,TRUE,ON_TIME,2026-06-25T21:14:00
1452,501116,C00074,2322000.0,3,C,ADS,Bandung,TRUE,ON_TIME,2026-06-18T03:42:00


In [9]:
q('''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT loan_id) AS distinct_loan_ids,
       COUNT(DISTINCT customer_id) AS distinct_customer_ids_raw
FROM loans;
''')

,n_rows,distinct_loan_ids,distinct_customer_ids_raw
0,1454,1448,809


## 2. Missing values in SQL

**How to think about it:** SQL has no built-in `isnull().sum()` — you build it yourself with a conditional `SUM`, one column at a time, all in a single row of output so it reads like a report.

In [10]:
q('''
SELECT
  SUM(CASE WHEN loan_amount IS NULL THEN 1 ELSE 0 END) AS null_loan_amount,
  SUM(CASE WHEN is_repeat_borrower IS NULL THEN 1 ELSE 0 END) AS null_is_repeat,
  SUM(CASE WHEN risk_tier IS NULL THEN 1 ELSE 0 END) AS null_risk_tier
FROM loans;
''')

,null_loan_amount,null_is_repeat,null_risk_tier
0,21,233,0


## 3. Duplicates in SQL

**How to think about it:** the exact same GROUP BY + HAVING pattern from the reference doc — but now it's catching a duplicate that's actually sitting in your data, not a hypothetical.

In [11]:
q('''
SELECT loan_id, COUNT(*) AS cnt
FROM loans
GROUP BY loan_id
HAVING COUNT(*) > 1;
''')

,loan_id,cnt
0,500200,2
1,500240,2
2,500354,2
3,500515,2
4,500572,2
5,500899,2


## 4. String normalization in SQL

**How to think about it:** pandas used `.strip().upper().replace()`. SQL's toolkit is `TRIM()`, `UPPER()`, `REPLACE()` — same operations, different syntax. First *measure* the mess (how many distinct raw values vs. cleaned values), then apply the fix.

In [12]:
# Before: how messy is it really?
q("SELECT COUNT(DISTINCT city) AS raw_distinct_cities FROM loans")

,raw_distinct_cities
0,20


In [13]:
# After: normalize and recount
q('''
SELECT COUNT(DISTINCT UPPER(TRIM(city))) AS clean_distinct_cities
FROM loans;
''')

,clean_distinct_cities
0,5


In [14]:
# Apply the same idea to customer_id, which also has a stray '-' to strip
q('''
SELECT UPPER(TRIM(REPLACE(customer_id, '-', ''))) AS customer_id_clean, COUNT(*) AS n
FROM loans
GROUP BY customer_id_clean
ORDER BY n DESC
LIMIT 10;
''')

,customer_id_clean,n
0,C00200,11
1,C00294,10
2,C00157,10
3,C00348,9
4,C00204,9
5,C00160,9
6,C00139,9
7,C00138,9
8,C00119,9
9,C00049,9


## 5. Range / sanity checks in SQL

**How to think about it:** `describe()`'s SQL equivalent is `MIN`/`MAX`/`AVG`, plus explicit conditional counts for the "impossible value" checks (zero or negative amounts/tenor) that `describe()` alone wouldn't flag for you.

In [15]:
q('''
SELECT MIN(loan_amount) AS min_amt, MAX(loan_amount) AS max_amt, AVG(loan_amount) AS avg_amt,
       SUM(CASE WHEN loan_amount <= 0 THEN 1 ELSE 0 END) AS non_positive_amt,
       SUM(CASE WHEN tenor_months <= 0 THEN 1 ELSE 0 END) AS non_positive_tenor
FROM loans;
''')

,min_amt,max_amt,avg_amt,non_positive_amt,non_positive_tenor
0,200000.0,8070000.0,991743.893929,0,2


## 6. GROUP BY — single dimension

**How to think about it:** the SQL mirror of `failed_df['status_reason'].value_counts()`. Establishes the overall picture before you cross dimensions.

In [16]:
q('''
SELECT repayment_status, COUNT(*) AS cnt
FROM loans
GROUP BY repayment_status
ORDER BY cnt DESC;
''')

,repayment_status,cnt
0,ON_TIME,1303
1,LATE,101
2,DEFAULTED,50


## 7. GROUP BY — combined dimensions

**How to think about it:** this is the step that's easy to skip and is exactly where the real signal lives. A "bad" outcome (LATE or DEFAULTED) isn't one category, so the conditional SUM groups them together as "not on time" while still letting you cut by two categorical dimensions at once.

In [17]:
q('''
SELECT acquisition_channel, risk_tier,
       SUM(CASE WHEN repayment_status != 'ON_TIME' THEN 1 ELSE 0 END) AS bad_count,
       COUNT(*) AS total
FROM loans
GROUP BY acquisition_channel, risk_tier
ORDER BY bad_count DESC;
''')

,acquisition_channel,risk_tier,bad_count,total
0,REFERRAL,A,18,147
1,ADS,A,17,137
2,APP,A,15,133
3,PARTNER_MERCHANT,C,14,69
4,PARTNER_MERCHANT,A,13,97
5,REFERRAL,B,12,121
6,APP,B,11,122
7,PARTNER_MERCHANT,B,9,112
8,ADS,C,8,67
9,APP,D,8,58


## 8. Rate, not raw count

**How to think about it:** same lesson as the Python side — a bigger `bad_count` might just mean a bigger group. Divide by that group's own total. `* 100.0` forces float division so it doesn't silently truncate to an integer.

In [18]:
q('''
SELECT acquisition_channel, risk_tier,
       SUM(CASE WHEN repayment_status != 'ON_TIME' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS bad_rate_pct,
       COUNT(*) AS total
FROM loans
GROUP BY acquisition_channel, risk_tier
ORDER BY bad_rate_pct DESC;
''')

,acquisition_channel,risk_tier,bad_rate_pct,total
0,PARTNER_MERCHANT,C,20.289855,69
1,APP,D,13.793103,58
2,PARTNER_MERCHANT,A,13.402062,97
3,ADS,A,12.408759,137
4,REFERRAL,A,12.244898,147
5,ADS,C,11.940299,67
6,REFERRAL,D,11.764706,51
7,APP,A,11.278195,133
8,REFERRAL,B,9.917355,121
9,ADS,D,9.259259,54


## 9. Recent window vs. baseline — the capstone diagnostic query

**How to think about it:** this is the SQL version of the pandas `comparison` dataframe from the EDA template — and it's the single query most likely to actually matter live, because "did something change recently" is the question behind every drop-diagnosis case. Build the cutoff once in a CTE so it's computed from the data itself (`MAX(disbursed_at) - 10 days`), not hardcoded. `NULLIF(..., 0)` guards against dividing by zero if any group has no rows in one of the two windows — a real edge case, worth having ready to explain if asked.

In [19]:
q('''
WITH bounds AS (
  SELECT date(MAX(disbursed_at), '-10 days') AS cutoff FROM loans
)
SELECT acquisition_channel, risk_tier,
  SUM(CASE WHEN date(disbursed_at) <= (SELECT cutoff FROM bounds)
            AND repayment_status != 'ON_TIME' THEN 1 ELSE 0 END) * 100.0 /
    NULLIF(SUM(CASE WHEN date(disbursed_at) <= (SELECT cutoff FROM bounds) THEN 1 ELSE 0 END), 0)
    AS baseline_bad_rate,
  SUM(CASE WHEN date(disbursed_at) > (SELECT cutoff FROM bounds)
            AND repayment_status != 'ON_TIME' THEN 1 ELSE 0 END) * 100.0 /
    NULLIF(SUM(CASE WHEN date(disbursed_at) > (SELECT cutoff FROM bounds) THEN 1 ELSE 0 END), 0)
    AS recent_bad_rate,
  COUNT(*) AS total
FROM loans
GROUP BY acquisition_channel, risk_tier
ORDER BY (recent_bad_rate - baseline_bad_rate) DESC;
''')

,acquisition_channel,risk_tier,baseline_bad_rate,recent_bad_rate,total
0,PARTNER_MERCHANT,C,5.769231,64.705882,69
1,PARTNER_MERCHANT,D,4.761905,18.181818,53
2,ADS,C,11.111111,15.384615,67
3,APP,C,2.000000,6.250000,66
4,PARTNER_MERCHANT,A,12.987013,15.000000,97
5,REFERRAL,C,6.000000,7.142857,64
6,ADS,B,5.063291,4.166667,103
7,APP,A,11.504425,10.000000,133
8,APP,D,14.285714,12.500000,58
9,REFERRAL,D,12.500000,9.090909,51


## 10. Window function bonus — ranking the result itself

**How to think about it:** window functions aren't only for per-row ranking inside raw tables — they can sit on top of an already-aggregated result too. Wrap the query above as a CTE, then `RANK()` the rows by how much the rate changed, so the worst offender is unambiguous rather than something you eyeball from a sorted list.

In [20]:
q('''
WITH bounds AS (
  SELECT date(MAX(disbursed_at), '-10 days') AS cutoff FROM loans
),
agg AS (
  SELECT acquisition_channel, risk_tier,
    SUM(CASE WHEN date(disbursed_at) <= (SELECT cutoff FROM bounds)
              AND repayment_status != 'ON_TIME' THEN 1 ELSE 0 END) * 100.0 /
      NULLIF(SUM(CASE WHEN date(disbursed_at) <= (SELECT cutoff FROM bounds) THEN 1 ELSE 0 END), 0)
      AS baseline_bad_rate,
    SUM(CASE WHEN date(disbursed_at) > (SELECT cutoff FROM bounds)
              AND repayment_status != 'ON_TIME' THEN 1 ELSE 0 END) * 100.0 /
      NULLIF(SUM(CASE WHEN date(disbursed_at) > (SELECT cutoff FROM bounds) THEN 1 ELSE 0 END), 0)
      AS recent_bad_rate
  FROM loans
  GROUP BY acquisition_channel, risk_tier
)
SELECT *,
       RANK() OVER (ORDER BY (recent_bad_rate - baseline_bad_rate) DESC) AS rate_change_rank
FROM agg
ORDER BY rate_change_rank;
''')

,acquisition_channel,risk_tier,baseline_bad_rate,recent_bad_rate,rate_change_rank
0,PARTNER_MERCHANT,C,5.769231,64.705882,1
1,PARTNER_MERCHANT,D,4.761905,18.181818,2
2,ADS,C,11.111111,15.384615,3
3,APP,C,2.000000,6.250000,4
4,PARTNER_MERCHANT,A,12.987013,15.000000,5
5,REFERRAL,C,6.000000,7.142857,6
6,ADS,B,5.063291,4.166667,7
7,APP,A,11.504425,10.000000,8
8,APP,D,14.285714,12.500000,9
9,REFERRAL,D,12.500000,9.090909,10


## Dialect cheat sheet — say this out loud if it comes up

SQLite (used here) is close to but not identical to what a company's real warehouse probably runs. If you're ever unsure or it matters, naming the dialect difference out loud is a good habit — it shows awareness, not uncertainty.

| Operation | SQLite | PostgreSQL | MySQL |
|---|---|---|---|
| Subtract days from a date | `date(x, '-10 days')` | `x - INTERVAL '10 days'` | `DATE_SUB(x, INTERVAL 10 DAY)` |
| String concatenation | `\|\|` | `\|\|` | `CONCAT(a, b)` |
| Limit rows | `LIMIT n` | `LIMIT n` | `LIMIT n` |
| Avoid divide-by-zero | `NULLIF(x, 0)` | `NULLIF(x, 0)` | `NULLIF(x, 0)` |
| Window functions (RANK, LAG, etc.) | Supported (3.25+) | Fully supported | Supported (8.0+) |

`CASE WHEN`, `GROUP BY`, `HAVING`, `NULLIF`, and window functions are portable across all three — that's most of what's in this notebook, which is why it's reasonably safe to assume the patterns transfer even if the exact syntax shifts slightly.

---

**Next step:** go back to your Python work on `loans_dataset.csv` and, for each pandas step you already wrote, write the SQL version next to it using this same `q()` setup. That pairing — not this notebook on its own — is the actual rehearsal for Monday.